# HW08-09: PyTorch MLP (S08-S09)

Содержание:
- загрузка и split датасета;
- обучение и сравнение E1-E4, O1-O3;
- сохранение всех обязательных артефактов в `artifacts/`.

In [1]:
import copy
import csv
import json
import os
import random
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Subset, random_split
from torchvision import datasets, transforms

SEED = 42
BATCH_SIZE = 128
BASE_EPOCHS = 12
EARLY_STOP_MAX_EPOCHS = 25
EARLY_STOP_PATIENCE = 4
O_SHORT_EPOCHS = 6
O3_EPOCHS = 12
MAX_TRAIN_SAMPLES = 24000
MAX_VAL_SAMPLES = 6000
MAX_TEST_SAMPLES = 10000

ROOT = Path('.').resolve()
DATA_DIR = ROOT / 'data'
ARTIFACTS_DIR = ROOT / 'artifacts'
FIGURES_DIR = ARTIFACTS_DIR / 'figures'


@dataclass
class ExperimentResult:
    experiment_id: str
    dataset: str
    seed: int
    model_summary: str
    optimizer: str
    lr: float
    momentum: float
    weight_decay: float
    epochs_trained: int
    best_val_accuracy: float
    best_val_loss: float
    best_epoch: int
    history: Dict[str, List[float]]
    best_state: Dict[str, torch.Tensor]
    model_config: Dict
    optimizer_config: Dict


class MLP(nn.Module):
    def __init__(
        self,
        input_dim: int,
        num_classes: int,
        hidden_sizes: List[int],
        dropout: float = 0.0,
        batchnorm: bool = False,
    ) -> None:
        super().__init__()
        layers: List[nn.Module] = []
        in_features = input_dim

        for hidden in hidden_sizes:
            layers.append(nn.Linear(in_features, hidden))
            if batchnorm:
                layers.append(nn.BatchNorm1d(hidden))
            layers.append(nn.ReLU())
            if dropout > 0:
                layers.append(nn.Dropout(dropout))
            in_features = hidden

        layers.append(nn.Linear(in_features, num_classes))

        self.flatten = nn.Flatten()
        self.net = nn.Sequential(*layers)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(self.flatten(x))


def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def get_device() -> torch.device:
    return torch.device('cuda' if torch.cuda.is_available() else 'cpu')


def load_first_available_dataset() -> Tuple[str, torch.utils.data.Dataset, torch.utils.data.Dataset]:
    grayscale_transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.5,), (0.5,)),
    ])
    rgb_transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
    ])

    errors: List[str] = []

    try:
        train_full = datasets.KMNIST(root=DATA_DIR, train=True, download=True, transform=grayscale_transform)
        test_set = datasets.KMNIST(root=DATA_DIR, train=False, download=True, transform=grayscale_transform)
        return 'KMNIST', train_full, test_set
    except Exception as e:
        errors.append(f'KMNIST: {e}')

    try:
        train_full = datasets.EMNIST(
            root=DATA_DIR,
            split='balanced',
            train=True,
            download=True,
            transform=grayscale_transform,
        )
        test_set = datasets.EMNIST(
            root=DATA_DIR,
            split='balanced',
            train=False,
            download=True,
            transform=grayscale_transform,
        )
        return 'EMNIST-balanced', train_full, test_set
    except Exception as e:
        errors.append(f'EMNIST-balanced: {e}')

    try:
        train_full = datasets.CIFAR10(root=DATA_DIR, train=True, download=True, transform=rgb_transform)
        test_set = datasets.CIFAR10(root=DATA_DIR, train=False, download=True, transform=rgb_transform)
        return 'CIFAR10', train_full, test_set
    except Exception as e:
        errors.append(f'CIFAR10: {e}')

    raise RuntimeError('Could not download any of KMNIST/EMNIST/CIFAR10. ' + ' | '.join(errors))


def prepare_data(seed: int) -> Tuple[str, DataLoader, DataLoader, DataLoader, int, int]:
    dataset_name, train_full, test_set = load_first_available_dataset()

    train_size = int(0.8 * len(train_full))
    val_size = len(train_full) - train_size

    generator = torch.Generator().manual_seed(seed)
    train_set, val_set = random_split(train_full, [train_size, val_size], generator=generator)

    if len(train_set) > MAX_TRAIN_SAMPLES:
        idx = torch.randperm(len(train_set), generator=generator)[:MAX_TRAIN_SAMPLES].tolist()
        train_set = Subset(train_set, idx)
    if len(val_set) > MAX_VAL_SAMPLES:
        idx = torch.randperm(len(val_set), generator=generator)[:MAX_VAL_SAMPLES].tolist()
        val_set = Subset(val_set, idx)
    if len(test_set) > MAX_TEST_SAMPLES:
        idx = torch.randperm(len(test_set), generator=generator)[:MAX_TEST_SAMPLES].tolist()
        test_set = Subset(test_set, idx)

    train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
    val_loader = DataLoader(val_set, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
    test_loader = DataLoader(test_set, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

    sample_x, _ = next(iter(train_loader))
    input_dim = int(np.prod(sample_x.shape[1:]))
    num_classes = len(train_full.classes)

    print(f'Dataset={dataset_name}; train={len(train_set)}; val={len(val_set)}; test={len(test_set)}', flush=True)
    return dataset_name, train_loader, val_loader, test_loader, input_dim, num_classes


def train_one_epoch(
    model: nn.Module,
    loader: DataLoader,
    criterion: nn.Module,
    optimizer: torch.optim.Optimizer,
    device: torch.device,
) -> Tuple[float, float]:
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for x, y in loader:
        x = x.to(device)
        y = y.to(device)

        optimizer.zero_grad()
        logits = model(x)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()

        batch_size = y.size(0)
        running_loss += loss.item() * batch_size
        preds = logits.argmax(dim=1)
        correct += (preds == y).sum().item()
        total += batch_size

    return running_loss / total, correct / total


def evaluate(
    model: nn.Module,
    loader: DataLoader,
    criterion: nn.Module,
    device: torch.device,
) -> Tuple[float, float]:
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for x, y in loader:
            x = x.to(device)
            y = y.to(device)

            logits = model(x)
            loss = criterion(logits, y)

            batch_size = y.size(0)
            running_loss += loss.item() * batch_size
            preds = logits.argmax(dim=1)
            correct += (preds == y).sum().item()
            total += batch_size

    return running_loss / total, correct / total


def build_optimizer(model: nn.Module, cfg: Dict) -> torch.optim.Optimizer:
    kind = cfg['name'].lower()
    lr = cfg['lr']
    weight_decay = cfg.get('weight_decay', 0.0)

    if kind == 'adam':
        return torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    if kind == 'sgd':
        momentum = cfg.get('momentum', 0.0)
        return torch.optim.SGD(model.parameters(), lr=lr, momentum=momentum, weight_decay=weight_decay)

    raise ValueError(f"Unsupported optimizer: {cfg['name']}")


def summarize_model(model_cfg: Dict) -> str:
    return (
        f"hidden={model_cfg['hidden_sizes']}; "
        f"act=ReLU; dropout={model_cfg.get('dropout', 0.0)}; "
        f"batchnorm={model_cfg.get('batchnorm', False)}"
    )

In [2]:
def run_experiment(
    experiment_id: str,
    dataset_name: str,
    seed: int,
    model_cfg: Dict,
    optimizer_cfg: Dict,
    epochs: int,
    train_loader: DataLoader,
    val_loader: DataLoader,
    device: torch.device,
    input_dim: int,
    num_classes: int,
    early_stopping_patience: Optional[int] = None,
) -> ExperimentResult:
    print(f'\n[START] {experiment_id}', flush=True)
    model = MLP(input_dim=input_dim, num_classes=num_classes, **model_cfg).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = build_optimizer(model, optimizer_cfg)

    history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}

    best_val_acc = -1.0
    best_val_loss = float('inf')
    best_epoch = -1
    best_state = copy.deepcopy(model.state_dict())
    epochs_no_improve = 0

    for epoch in range(1, epochs + 1):
        train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
        val_loss, val_acc = evaluate(model, val_loader, criterion, device)

        print(
            f"{experiment_id} | epoch {epoch}/{epochs} | "
            f"train_loss={train_loss:.4f} train_acc={train_acc:.4f} | "
            f"val_loss={val_loss:.4f} val_acc={val_acc:.4f}",
            flush=True,
        )

        history['train_loss'].append(train_loss)
        history['train_acc'].append(train_acc)
        history['val_loss'].append(val_loss)
        history['val_acc'].append(val_acc)

        improved = val_acc > best_val_acc
        if improved:
            best_val_acc = val_acc
            best_val_loss = val_loss
            best_epoch = epoch
            best_state = copy.deepcopy(model.state_dict())
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1

        if early_stopping_patience is not None and epochs_no_improve >= early_stopping_patience:
            print(f"{experiment_id} | early stop at epoch {epoch} (patience={early_stopping_patience})", flush=True)
            break

    print(
        f"[DONE] {experiment_id}: best_val_acc={best_val_acc:.4f}, best_val_loss={best_val_loss:.4f}, best_epoch={best_epoch}",
        flush=True,
    )

    return ExperimentResult(
        experiment_id=experiment_id,
        dataset=dataset_name,
        seed=seed,
        model_summary=summarize_model(model_cfg),
        optimizer=optimizer_cfg['name'],
        lr=float(optimizer_cfg['lr']),
        momentum=float(optimizer_cfg.get('momentum', 0.0)),
        weight_decay=float(optimizer_cfg.get('weight_decay', 0.0)),
        epochs_trained=len(history['val_acc']),
        best_val_accuracy=float(best_val_acc),
        best_val_loss=float(best_val_loss),
        best_epoch=best_epoch,
        history=history,
        best_state=best_state,
        model_config=model_cfg,
        optimizer_config=optimizer_cfg,
    )


def save_runs_csv(results: List[ExperimentResult], path: Path) -> None:
    fieldnames = [
        'experiment_id',
        'dataset',
        'seed',
        'model_summary',
        'optimizer',
        'lr',
        'momentum',
        'weight_decay',
        'epochs_trained',
        'best_val_accuracy',
        'best_val_loss',
    ]
    with path.open('w', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        for r in results:
            writer.writerow(
                {
                    'experiment_id': r.experiment_id,
                    'dataset': r.dataset,
                    'seed': r.seed,
                    'model_summary': r.model_summary,
                    'optimizer': r.optimizer,
                    'lr': r.lr,
                    'momentum': r.momentum,
                    'weight_decay': r.weight_decay,
                    'epochs_trained': r.epochs_trained,
                    'best_val_accuracy': f"{r.best_val_accuracy:.6f}",
                    'best_val_loss': f"{r.best_val_loss:.6f}",
                }
            )


def plot_best_curves(history: Dict[str, List[float]], out_path: Path) -> None:
    epochs = range(1, len(history['train_loss']) + 1)
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    axes[0].plot(epochs, history['train_loss'], label='train_loss')
    axes[0].plot(epochs, history['val_loss'], label='val_loss')
    axes[0].set_title('E4 Loss')
    axes[0].set_xlabel('epoch')
    axes[0].set_ylabel('loss')
    axes[0].legend()

    axes[1].plot(epochs, history['train_acc'], label='train_acc')
    axes[1].plot(epochs, history['val_acc'], label='val_acc')
    axes[1].set_title('E4 Accuracy')
    axes[1].set_xlabel('epoch')
    axes[1].set_ylabel('accuracy')
    axes[1].legend()

    fig.tight_layout()
    fig.savefig(out_path, dpi=150)
    plt.close(fig)


def plot_lr_extremes(o1_history: Dict[str, List[float]], o2_history: Dict[str, List[float]], out_path: Path) -> None:
    e1 = range(1, len(o1_history['val_loss']) + 1)
    e2 = range(1, len(o2_history['val_loss']) + 1)

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    axes[0].plot(e1, o1_history['val_loss'], marker='o', label='O1 val_loss (lr=1e-1)')
    axes[0].plot(e2, o2_history['val_loss'], marker='o', label='O2 val_loss (lr=1e-5)')
    axes[0].set_title('Val Loss: LR Extremes')
    axes[0].set_xlabel('epoch')
    axes[0].set_ylabel('loss')
    axes[0].legend()

    axes[1].plot(e1, o1_history['val_acc'], marker='o', label='O1 val_acc (lr=1e-1)')
    axes[1].plot(e2, o2_history['val_acc'], marker='o', label='O2 val_acc (lr=1e-5)')
    axes[1].set_title('Val Accuracy: LR Extremes')
    axes[1].set_xlabel('epoch')
    axes[1].set_ylabel('accuracy')
    axes[1].legend()

    fig.tight_layout()
    fig.savefig(out_path, dpi=150)
    plt.close(fig)


def main() -> Dict:
    set_seed(SEED)
    torch.set_num_threads(min(8, os.cpu_count() or 8))
    ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)
    FIGURES_DIR.mkdir(parents=True, exist_ok=True)

    device = get_device()
    dataset_name, train_loader, val_loader, test_loader, input_dim, num_classes = prepare_data(SEED)

    base_model_cfg = {'hidden_sizes': [512, 256, 128], 'dropout': 0.0, 'batchnorm': False}

    e1 = run_experiment(
        'E1', dataset_name, SEED, base_model_cfg,
        {'name': 'Adam', 'lr': 1e-3, 'weight_decay': 0.0},
        BASE_EPOCHS, train_loader, val_loader, device, input_dim, num_classes,
    )

    e2_model_cfg = {**base_model_cfg, 'dropout': 0.3}
    e2 = run_experiment(
        'E2', dataset_name, SEED, e2_model_cfg,
        {'name': 'Adam', 'lr': 1e-3, 'weight_decay': 0.0},
        BASE_EPOCHS, train_loader, val_loader, device, input_dim, num_classes,
    )

    e3_model_cfg = {**base_model_cfg, 'batchnorm': True}
    e3 = run_experiment(
        'E3', dataset_name, SEED, e3_model_cfg,
        {'name': 'Adam', 'lr': 1e-3, 'weight_decay': 0.0},
        BASE_EPOCHS, train_loader, val_loader, device, input_dim, num_classes,
    )

    best_prev = e2 if e2.best_val_accuracy >= e3.best_val_accuracy else e3
    e4 = run_experiment(
        'E4', dataset_name, SEED, best_prev.model_config, best_prev.optimizer_config,
        EARLY_STOP_MAX_EPOCHS, train_loader, val_loader, device, input_dim, num_classes,
        early_stopping_patience=EARLY_STOP_PATIENCE,
    )

    o1 = run_experiment(
        'O1', dataset_name, SEED, e4.model_config,
        {'name': 'Adam', 'lr': 1e-1, 'weight_decay': 0.0},
        O_SHORT_EPOCHS, train_loader, val_loader, device, input_dim, num_classes,
    )
    o2 = run_experiment(
        'O2', dataset_name, SEED, e4.model_config,
        {'name': 'Adam', 'lr': 1e-5, 'weight_decay': 0.0},
        O_SHORT_EPOCHS, train_loader, val_loader, device, input_dim, num_classes,
    )
    o3 = run_experiment(
        'O3', dataset_name, SEED, e4.model_config,
        {'name': 'SGD', 'lr': 5e-3, 'momentum': 0.9, 'weight_decay': 1e-4},
        O3_EPOCHS, train_loader, val_loader, device, input_dim, num_classes,
    )

    best_model = MLP(input_dim=input_dim, num_classes=num_classes, **e4.model_config).to(device)
    best_model.load_state_dict(e4.best_state)
    criterion = nn.CrossEntropyLoss()
    test_loss, test_acc = evaluate(best_model, test_loader, criterion, device)

    torch.save(e4.best_state, ARTIFACTS_DIR / 'best_model.pt')

    best_config = {
        'dataset': dataset_name,
        'seed': SEED,
        'device': str(device),
        'model': e4.model_config,
        'optimizer': e4.optimizer_config,
        'early_stopping_patience': EARLY_STOP_PATIENCE,
        'epochs_trained': e4.epochs_trained,
        'best_epoch': e4.best_epoch,
        'best_val_accuracy': e4.best_val_accuracy,
        'best_val_loss': e4.best_val_loss,
        'test_accuracy': test_acc,
        'test_loss': test_loss,
    }
    (ARTIFACTS_DIR / 'best_config.json').write_text(json.dumps(best_config, indent=2), encoding='utf-8')

    all_results = [e1, e2, e3, e4, o1, o2, o3]
    save_runs_csv(all_results, ARTIFACTS_DIR / 'runs.csv')

    plot_best_curves(e4.history, FIGURES_DIR / 'curves_best.png')
    plot_lr_extremes(o1.history, o2.history, FIGURES_DIR / 'curves_lr_extremes.png')

    summary = {
        'device': str(device),
        'dataset': dataset_name,
        'best_in_A': 'E4',
        'picked_from': best_prev.experiment_id,
        'best_val_accuracy': e4.best_val_accuracy,
        'best_val_loss': e4.best_val_loss,
        'test_accuracy': test_acc,
        'test_loss': test_loss,
        'o1_best_val_accuracy': o1.best_val_accuracy,
        'o2_best_val_accuracy': o2.best_val_accuracy,
        'o3_best_val_accuracy': o3.best_val_accuracy,
    }
    print(json.dumps(summary, indent=2))
    return summary

In [3]:
set_seed(SEED)
print('seed:', SEED)
print('device:', get_device())
print('torch:', torch.__version__)
print('torchvision:', __import__('torchvision').__version__)

dataset_name, train_loader, val_loader, test_loader, input_dim, num_classes = prepare_data(SEED)
x, y = next(iter(train_loader))
print('dataset:', dataset_name)
print('input_dim:', input_dim, 'num_classes:', num_classes)
print('batch x shape:', tuple(x.shape), 'batch y shape:', tuple(y.shape))
print('x range:', float(x.min()), float(x.max()))

seed: 42
device: cpu
torch: 2.10.0+cpu
torchvision: 0.25.0+cpu
Dataset=EMNIST-balanced; train=24000; val=6000; test=10000
dataset: EMNIST-balanced
input_dim: 784 num_classes: 47
batch x shape: (128, 1, 28, 28) batch y shape: (128,)
x range: -1.0 1.0


In [4]:
summary = main()
summary

Dataset=EMNIST-balanced; train=24000; val=6000; test=10000

[START] E1
E1 | epoch 1/12 | train_loss=1.9743 train_acc=0.4498 | val_loss=1.2676 val_acc=0.6222
E1 | epoch 2/12 | train_loss=1.0568 train_acc=0.6817 | val_loss=0.9672 val_acc=0.6998
E1 | epoch 3/12 | train_loss=0.8255 train_acc=0.7420 | val_loss=0.8312 val_acc=0.7388
E1 | epoch 4/12 | train_loss=0.6966 train_acc=0.7759 | val_loss=0.7323 val_acc=0.7728
E1 | epoch 5/12 | train_loss=0.6195 train_acc=0.7937 | val_loss=0.7230 val_acc=0.7623
E1 | epoch 6/12 | train_loss=0.5506 train_acc=0.8143 | val_loss=0.7203 val_acc=0.7653
E1 | epoch 7/12 | train_loss=0.4982 train_acc=0.8282 | val_loss=0.6794 val_acc=0.7740
E1 | epoch 8/12 | train_loss=0.4432 train_acc=0.8442 | val_loss=0.6616 val_acc=0.7868
E1 | epoch 9/12 | train_loss=0.4169 train_acc=0.8524 | val_loss=0.6444 val_acc=0.7928
E1 | epoch 10/12 | train_loss=0.3835 train_acc=0.8604 | val_loss=0.6535 val_acc=0.7917
E1 | epoch 11/12 | train_loss=0.3554 train_acc=0.8683 | val_loss=0.6

{'device': 'cpu',
 'dataset': 'EMNIST-balanced',
 'best_in_A': 'E4',
 'picked_from': 'E3',
 'best_val_accuracy': 0.8078333333333333,
 'best_val_loss': 0.6371665371259053,
 'test_accuracy': 0.8046,
 'test_loss': 0.6402625736236572,
 'o1_best_val_accuracy': 0.7556666666666667,
 'o2_best_val_accuracy': 0.5063333333333333,
 'o3_best_val_accuracy': 0.8076666666666666}

In [5]:
import pandas as pd

runs = pd.read_csv(Path('artifacts') / 'runs.csv')
runs

,experiment_id,dataset,seed,model_summary,optimizer,lr,momentum,weight_decay,epochs_trained,best_val_accuracy,best_val_loss
0,E1,EMNIST-balanced,42,"hidden=[512, 256, 128]; act=ReLU; dropout=0.0;...",Adam,0.00100,0.0,0.0000,12,0.801833,0.645443
1,E2,EMNIST-balanced,42,"hidden=[512, 256, 128]; act=ReLU; dropout=0.3;...",Adam,0.00100,0.0,0.0000,12,0.790333,0.638896
2,E3,EMNIST-balanced,42,"hidden=[512, 256, 128]; act=ReLU; dropout=0.0;...",Adam,0.00100,0.0,0.0000,12,0.809000,0.602532
3,E4,EMNIST-balanced,42,"hidden=[512, 256, 128]; act=ReLU; dropout=0.0;...",Adam,0.00100,0.0,0.0000,13,0.807833,0.637167
4,O1,EMNIST-balanced,42,"hidden=[512, 256, 128]; act=ReLU; dropout=0.0;...",Adam,0.10000,0.0,0.0000,6,0.755667,0.844366
5,O2,EMNIST-balanced,42,"hidden=[512, 256, 128]; act=ReLU; dropout=0.0;...",Adam,0.00001,0.0,0.0000,6,0.506333,2.691070
6,O3,EMNIST-balanced,42,"hidden=[512, 256, 128]; act=ReLU; dropout=0.0;...",SGD,0.00500,0.9,0.0001,12,0.807667,0.636585


In [6]:
best_config = json.loads((Path('artifacts') / 'best_config.json').read_text(encoding='utf-8'))
best_config

{'dataset': 'EMNIST-balanced',
 'seed': 42,
 'device': 'cpu',
 'model': {'hidden_sizes': [512, 256, 128], 'dropout': 0.0, 'batchnorm': True},
 'optimizer': {'name': 'Adam', 'lr': 0.001, 'weight_decay': 0.0},
 'early_stopping_patience': 4,
 'epochs_trained': 13,
 'best_epoch': 9,
 'best_val_accuracy': 0.8078333333333333,
 'best_val_loss': 0.6371665371259053,
 'test_accuracy': 0.8046,
 'test_loss': 0.6402625736236572}